# Example 11.3.2 / 5: Keep Linear Parts Visible with linelim

Companion notebook for **11.3.2 Substitution and Expression Expansion** in the AMPL lab Documentation, section "Keep Linear Parts Visible with linelim". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Keep Linear Parts Visible with linelim

The `linelim` option controls the treatment of linear parts of defined expressions. With the
default `linelim 1`, a linear part is moved into the constraints that use the defined variable,
and only the nonlinear part, written `Cost.nl` by `solexpand`, stays defined; `linelim 0` leaves
the whole defining expression represented as nonlinear. The defined variable `Cost` below has the
linear part `3*Other`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Ship >= 0, <= 10;                                 # shipment
var Other >= 0, <= 10;                                # second activity
var Cost = 3 * Other + (10 + 4 * Ship) / (1 + Ship);  # defined variable with linear part 3 * Other

minimize Total: Cost * Ship + Other;  # total cost

subject to Limit: Cost + Ship <= 20;  # row shown by solexpand

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for value in (1, 0):
    ampl.option["linelim"] = value
    print("linelim", value)
    print(ampl.get_output("solexpand Limit;"), end="")

linelim 1
subject to Limit:
	Cost.nl + Ship + 3*Other <= 20;

linelim 0
subject to Limit:
	Cost + Ship <= 20;



### Pure AMPL: model.run

Pure AMPL, for `linelim 0`, with `expand` for comparison:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option linelim 0;
solexpand Limit;
expand Limit;

subject to Limit:
	Cost + Ship <= 20;

subject to Limit:
	Ship + Cost <= 20;

